In [1]:
import json
import time
import re
import sys
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel
from tqdm.auto import tqdm

load_dotenv(Path('../../.env'))

# add app to path
sys.path.append('../app')
import ingest
import search_agent
import logs

In [2]:
# load indexes
index, vindex, embedding_model = ingest.index_data(
    'DataTalksClub',
    'data-engineering-zoomcamp'
)

# initialize agent
agent = search_agent.init_agent(
    index, vindex, embedding_model,
    'DataTalksClub',
    'data-engineering-zoomcamp'
)

# load questions
with open('questions.json', 'r') as f:
    questions = json.load(f)

print(f"Loaded {len(questions)} questions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 10 questions


In [3]:
for q in tqdm(questions):
    print(q)
    result = await agent.run(user_prompt=q)
    print(result.output)
    logs.log_interaction_to_file(agent, result.new_messages(), source='ai-generated')
    print()
    time.sleep(30)

  0%|          | 0/10 [00:00<?, ?it/s]

What is the purpose of the 'zone' table in the SQL query provided for Question 5?
<function=search {"query": "Zone table purpose in Question 5 SQL query"} </function>

How do I submit ad-hoc queries against my connections using 'bruin query'?
You can submit ad-hoc queries against your connections using 'bruin query' by running the command `bruin query --connection <conn> --query "..."` in your terminal. Replace `<conn>` with the name of your connection and `"...` with your SQL query. 

For example: 
```bash
bruin query --connection duckdb-default \
  --query "SELECT * FROM ingestion.trips LIMIT 10"
```
This will execute the SQL query `SELECT * FROM ingestion.trips LIMIT 10` against the `duckdb-default` connection.

For more information, you can refer to the [bruin query documentation](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/05-data-platforms/notes/06-core-05-commands.md).

What is the purpose of Prefect Secret blocks and how do I create one?
The purpose of 

In [4]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


In [5]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked? 

Output true/false for each check and provide a short explanation for your judgment.
""".strip()



In [6]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [7]:
eval_agent = Agent(
    name='eval_agent',
    model=GroqModel('llama-3.3-70b-versatile'),
    instructions=evaluation_prompt + "\n\nReturn ONLY a JSON object with keys 'checklist' and 'summary'. No tags, no markdown."
)

In [8]:
def simplify_log_messages(messages):
    log_simplified = []
    for m in messages:
        parts = []
        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']
            if kind == 'user-prompt':
                part.pop('timestamp', None)
            if kind == 'tool-call':
                part.pop('tool_call_id', None)
            if kind == 'tool-return':
                part.pop('tool_call_id', None)
                part.pop('metadata', None)
                part.pop('timestamp', None)
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                part.pop('id', None)
            parts.append(part)
        log_simplified.append({'kind': m['kind'], 'parts': parts})
    return log_simplified

In [26]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']
    instructions = log_record['system_prompt']
    question = messages[0]['parts'][1]['content']
    answer = messages[-1]['parts'][0]['content']
    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt)
    raw = result.output
    raw = re.sub(r'<function[^>]*>', '', raw)
    raw = re.sub(r'</function>', '', raw).strip()
    
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        data = json.loads(json_match.group())
        
        # Handle checklist as dict: {"checklist": {"instructions_follow": true, ...}}
        if 'checklist' in data and isinstance(data['checklist'], dict):
            checklist = [
                EvaluationCheck(
                    check_name=k,
                    justification="",
                    check_pass=v
                )
                for k, v in data['checklist'].items()
                if isinstance(v, bool)
            ]
            
            # handle summary being either a string or a dict
            summary = data.get('summary', '')
            if isinstance(summary, dict):
                summary = summary.get('explanation', '')
            
            return EvaluationChecklist(
                checklist=checklist,
                summary=summary
            )
        
        # Handle flat format: {"instructions_follow": true, ...}
        if 'checklist' not in data:
            checklist = [
                EvaluationCheck(
                    check_name=k,
                    justification="",
                    check_pass=v
                )
                for k, v in data.items()
                if isinstance(v, bool)
            ]
            return EvaluationChecklist(checklist=checklist, summary="")
    
    return EvaluationChecklist(**data)

In [27]:
import importlib
importlib.reload(logs)

def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

LOG_DIR = Path('logs')

eval_set = [
    load_log_file(f)
    for f in LOG_DIR.iterdir()
    if load_log_file(f)['source'] == 'ai-generated'
]

print(f"Found {len(eval_set)} logs")


Found 10 logs


In [29]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))
    time.sleep(10)

print(f"Evaluated {len(eval_results)} logs")

  0%|          | 0/10 [00:00<?, ?it/s]

Evaluated 10 logs


In [28]:
rows = []
for log_record, eval_result in eval_results:
    messages = log_record['messages']
    row = {
        'question': messages[0]['parts'][1]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }
    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df_evals = pd.DataFrame(rows)
print(df_evals.mean(numeric_only=True))

instructions_follow    0.7
instructions_avoid     1.0
answer_relevant        0.5
answer_clear           0.5
answer_citations       0.4
completeness           0.4
tool_call_search       1.0
dtype: float64


In [25]:
# check structure of first log
log_record = eval_set[0]
messages = log_record['messages']
print("Number of messages:", len(messages))
for i, m in enumerate(messages):
    print(f"\nMessage {i}: kind={m['kind']}")
    for p in m['parts']:
        print(f"  part_kind={p['part_kind']}, content preview={str(p.get('content', ''))[:100]}")

Number of messages: 6

Message 0: kind=request
  part_kind=system-prompt, content preview=You are a helpful assistant for data-engineering-zoomcamp by DataTalksClub.
Use the search tool to f
  part_kind=user-prompt, content preview=What is the difference between a pass-through Flink job and a regular Flink job?

Message 1: kind=response
  part_kind=tool-call, content preview=

Message 2: kind=request
  part_kind=tool-return, content preview=[{'filename': '07-streaming/workshop/README.md', 'section': '## The pass-through Flink job\n\nNow le

Message 3: kind=response
  part_kind=text, content preview=Unfortunately, the search results don't provide a clear answer to the question. Let me try a differe
  part_kind=tool-call, content preview=

Message 4: kind=request
  part_kind=tool-return, content preview=[{'filename': '07-streaming/workshop/README.md', 'section': '## Q&A\n\nQuestions and answers from th

Message 5: kind=response
  part_kind=text, content preview=The difference between a p